- https://docs.planet.com/develop/apis/data/


In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os
import json
import requests
import geojsonio
import time

load_dotenv()

True

In [ ]:
def p(data):
    print(json.dumps(data, indent=2))

# Connect


In [37]:
if os.environ.get('PL_API_KEY', ''):
    API_KEY = os.environ.get('PL_API_KEY', '')
else:
    API_KEY = 'PASTE_YOUR_API_KEY_HERE'

BASIC_AUTH = (API_KEY, '')

In [38]:
URL = "https://api.planet.com/data/v1"

session = requests.Session()
session.auth = (API_KEY, "")
res = session.get(URL)

In [39]:
res.status_code

200

In [40]:
# res.text

In [41]:
p(res.json())

{
  "_links": {
    "_self": "https://api.planet.com/data/v1/",
    "asset-types": "https://api.planet.com/data/v1/asset-types/",
    "item-types": "https://api.planet.com/data/v1/item-types/",
    "spec": "https://api.planet.com/data/v1/spec"
  }
}


# Filter


In [16]:
stats_url = "{}/stats".format(URL)
print(stats_url)

https://api.planet.com/data/v1/stats


In [ ]:
date_filter = {
    "type": "DateRangeFilter", # Type of filter -> Date Range
    "field_name": "acquired", # The field to filter on: "acquired" -> Date on which the "image was taken"
    "config": {
        "gte": "2020-01-01T00:00:00.000Z", # "gte" -> Greater than or equal to
    }
}

request = {
    "item_types" : ["PSScene"],
    "interval" : "year", # aggregate results by year
    "filter" : date_filter
}

# Send the POST request to the API stats endpoint
res = session.post(stats_url, json=request)

# Print response
p(res.json())

In [ ]:
# Search for imagery only from PlanetScope satellites that have a PS2 telescope

item_types = ["PSScene"]

instrument_filter = {
    "type": "StringInFilter",
    "field_name": "instrument",
    "config": ["PS2"]
}

request = {
    "item_types" : item_types,
    "interval" : "year",
    "filter" : instrument_filter
}

res = session.post(stats_url, json=request)

p(res.json())

In [ ]:
# Search for imagery that only intersects with 40N, 90W

geom = {
    "type": "Point",
    "coordinates": [
        -90,
         40
    ]
}

geometry_filter = {
    "type": "GeometryFilter",
    "field_name": "geometry",
    "config": geom
}

request = {
    "item_types" : item_types,
    "interval" : "year",
    "filter" : geometry_filter
}

res=session.post(stats_url, json=request)

p(res.json())

In [ ]:
and_filter = {
    "type": "AndFilter",
    "config": [instrument_filter, geometry_filter, date_filter]
}

# Print the logical filter
p(and_filter)

request = {
    "item_types" : item_types,
    "interval" : "year",
    "filter" : and_filter
}

res=session.post(stats_url, json=request)

p(res.json())

# Quick Search


In [17]:
quick_url = "{}/quick-search".format(URL)
print(quick_url)

https://api.planet.com/data/v1/quick-search


In [18]:
item_types = ["PSScene"]

geom = {
    "type": "Point",
    "coordinates": [
        -90,
         40
    ]
}

geometry_filter = {
    "type": "GeometryFilter",
    "field_name": "geometry",
    "config": geom
}

request = {
    "item_types" : item_types,
    "filter" : geometry_filter
}

res = session.post(quick_url, json=request)

geojson = res.json()

p(geojson)

{
  "_links": {
    "_first": "https://api.planet.com/data/v1/searches/404292e31360413982b837c4acbe6a21/results?_page=eyJwYWdlX3NpemUiOiAyNTAsICJzb3J0X2J5IjogInB1Ymxpc2hlZCIsICJzb3J0X2Rlc2MiOiB0cnVlLCAic29ydF9wcmV2IjogZmFsc2UsICJxdWVyeV9wYXJhbXMiOiB7fX0%3D",
    "_next": "https://api.planet.com/data/v1/searches/404292e31360413982b837c4acbe6a21/results?_page=eyJwYWdlX3NpemUiOiAyNTAsICJzb3J0X2J5IjogInB1Ymxpc2hlZCIsICJzb3J0X2Rlc2MiOiB0cnVlLCAic29ydF9zdGFydCI6ICIyMDI1LTEyLTI5VDIwOjU5OjI5LjAwMDAwMFoiLCAic29ydF9sYXN0X2lkIjogIjIwMjUxMjI5XzE3MTYwMF83NV8yNGRhIiwgInNvcnRfcHJldiI6IGZhbHNlLCAicXVlcnlfcGFyYW1zIjoge319",
    "_self": "https://api.planet.com/data/v1/searches/404292e31360413982b837c4acbe6a21/results?_page=eyJwYWdlX3NpemUiOiAyNTAsICJzb3J0X2J5IjogInB1Ymxpc2hlZCIsICJzb3J0X2Rlc2MiOiB0cnVlLCAic29ydF9wcmV2IjogZmFsc2UsICJxdWVyeV9wYXJhbXMiOiB7fX0%3D"
  },
  "features": [
    {
      "_links": {
        "_self": "https://api.planet.com/data/v1/item-types/PSScene/items/20260608_164951_05_256c",

In [19]:
features = geojson["features"]

# Get the number of features present in the response
len(features)

250

In [20]:
for f in features:
    # Print the ID for each feature
    p(f["id"])

"20260608_164951_05_256c"
"20260409_165009_53_2550"
"20251218_165238_00_2550"
"20260607_165216_63_2575"
"20260606_172354_27_2534"
"20260606_172356_50_2534"
"20260605_172315_95_2530"
"20260605_165211_39_254c"
"20260604_172504_00_2526"
"20260604_172506_24_2526"
"20260602_172249_30_2531"
"20260602_172247_06_2531"
"20260601_165006_06_255f"
"20260529_172214_89_253c"
"20260529_172217_11_253c"
"20260529_172551_56_2527"
"20260527_165138_82_254d"
"20260527_165136_46_254d"
"20260526_173144_40_2523"
"20260526_173142_35_2523"
"20260525_172129_14_254a"
"20260524_172328_28_252d"
"20260524_172330_51_252d"
"20260523_172150_91_252e"
"20260522_164912_14_2562"
"20260522_172433_20_2527"
"20260521_172128_13_253a"
"20260521_165339_20_2520"
"20260521_165341_55_2520"
"20260519_180201_26_24f5"
"20260518_165011_02_2555"
"20260518_165008_66_2555"
"20260517_164808_58_2570"
"20260517_164806_22_2570"
"20260516_173005_70_2501"
"20260516_173111_53_2514"
"20260514_173122_52_250b"
"20260513_165300_75_257a"
"20260512_17

In [21]:
# When the number of matching items exceeds 250, the results are delivered in pages. Let's perform a search query that should return a large number of results:

## Pagination Query


In [33]:
geom = {
    "type": "Polygon",
    "coordinates": [
      [
        [
          -125.29632568359376,
          48.37084770238366
        ],
        [
          -125.29632568359376,
          49.335861591104106
        ],
        [
          -123.2391357421875,
          49.335861591104106
        ],
        [
          -123.2391357421875,
          48.37084770238366
        ],
        [
          -125.29632568359376,
          48.37084770238366
        ]
      ]
    ]
  }

# Setup the geometry filter
geometry_filter = {
    "type": "GeometryFilter",
    "field_name": "geometry",
    "config": geom
}

# Setup the request
request = {
    "item_types" : item_types,
    "filter" : geometry_filter
}

In [52]:
res = session.post(quick_url, json=request)

geojson = res.json()
len(geojson["features"])

250

In [53]:
p(geojson["_links"])

{
  "_first": "https://api.planet.com/data/v1/searches/512b5ec4cb04400da95ae2caf639e04c/results?_page=eyJwYWdlX3NpemUiOiAyNTAsICJzb3J0X2J5IjogInB1Ymxpc2hlZCIsICJzb3J0X2Rlc2MiOiB0cnVlLCAic29ydF9wcmV2IjogZmFsc2UsICJxdWVyeV9wYXJhbXMiOiB7fX0%3D",
  "_next": "https://api.planet.com/data/v1/searches/512b5ec4cb04400da95ae2caf639e04c/results?_page=eyJwYWdlX3NpemUiOiAyNTAsICJzb3J0X2J5IjogInB1Ymxpc2hlZCIsICJzb3J0X2Rlc2MiOiB0cnVlLCAic29ydF9zdGFydCI6ICIyMDI2LTA2LTA0VDIxOjU5OjAxLjAwMDAwMFoiLCAic29ydF9sYXN0X2lkIjogIjIwMjYwNjA0XzE5MTY0Nl80OV8yNTYyIiwgInNvcnRfcHJldiI6IGZhbHNlLCAicXVlcnlfcGFyYW1zIjoge319",
  "_self": "https://api.planet.com/data/v1/searches/512b5ec4cb04400da95ae2caf639e04c/results?_page=eyJwYWdlX3NpemUiOiAyNTAsICJzb3J0X2J5IjogInB1Ymxpc2hlZCIsICJzb3J0X2Rlc2MiOiB0cnVlLCAic29ydF9wcmV2IjogZmFsc2UsICJxdWVyeV9wYXJhbXMiOiB7fX0%3D"
}


In [54]:
next_url = geojson["_links"]["_next"]

# Print the link to the next page of results
print(next_url)

https://api.planet.com/data/v1/searches/512b5ec4cb04400da95ae2caf639e04c/results?_page=eyJwYWdlX3NpemUiOiAyNTAsICJzb3J0X2J5IjogInB1Ymxpc2hlZCIsICJzb3J0X2Rlc2MiOiB0cnVlLCAic29ydF9zdGFydCI6ICIyMDI2LTA2LTA0VDIxOjU5OjAxLjAwMDAwMFoiLCAic29ydF9sYXN0X2lkIjogIjIwMjYwNjA0XzE5MTY0Nl80OV8yNTYyIiwgInNvcnRfcHJldiI6IGZhbHNlLCAicXVlcnlfcGFyYW1zIjoge319


In [55]:
features = geojson["features"]

# Get the first result's feature
feature = features[0]

# Print the ID
p(feature["id"])

# Print the permissions
p(feature["_permissions"])

"20260609_200236_58_251a"
[]


In [56]:
# Get the assets link for the item
assets_url = feature["_links"]["assets"]

# Print the assets link
print(assets_url)

https://api.planet.com/data/v1/item-types/PSScene/items/20260609_200236_58_251a/assets/


In [57]:
# Send a GET request to the assets url for the item (Get the list of available assets for the item)
res = session.get(assets_url)

# Assign a variable to the response
assets = res.json()

In [58]:
# Print the asset types that are available
print(assets.keys())

dict_keys([])
